# 02 - Model vs. measurement

Before calibrating anything, it helps to see *how far off* an uncalibrated model
is. This notebook

1. builds a small plant in a few lines,
2. creates a measurement record from that same plant with a known parameter
   value ("twin data"),
3. simulates with the model's default parameters and plots both against each
   other.

Because we generated the data ourselves, we know the answer any calibration
would have to find.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from demo_plant import (
    DEFAULT_K_HYD_CH,
    TRUE_PARAMETERS,
    build_demo_plant,
    make_twin_measurements,
    simulate,
)

print("true parameter used to generate the data:", TRUE_PARAMETERS)
print("model default the simulation starts from:", DEFAULT_K_HYD_CH)

## The plant

One digester, one CHP - that is all `build_demo_plant()` does:

```python
cfg.add_digester("F1", V_liq=2000, V_gas=400, T_ad=313.15, Q_substrates=[15, 10, 0])
cfg.add_chp("chp", P_el_nom=500.0, eta_el=0.40, eta_th=0.45)
cfg.auto_connect_digester_to_chp("F1", "chp")
```

One rule is easy to miss: **the plant must declare exactly as many substrates as
the measurement frame has feed columns**. Our frame carries three
(`Q_sub_maize`, `Q_sub_manure`, `Q_sub_grass`), so the feedstock lists three.
A mismatch raises `ValueError: Q has 3 entries ...`.

In [ ]:
plant = build_demo_plant(days=5)
print(f"components: {[c for c in plant.components]}")

## The measurements

`make_twin_measurements()` simulates this plant with `k_hyd_ch = 2.0` and adds
2 % noise. That is our stand-in for a plant archive, with the advantage that
the "true" parameter is known.

In [ ]:
measurements = make_twin_measurements(days=5, noise=0.02, seed=0)
measurements.data[["Q_gas", "Q_ch4", "P_el"]].describe().round(1)

## Simulating with the default parameters

In [ ]:
default_k = DEFAULT_K_HYD_CH   # what an uncalibrated model uses
predicted = simulate(plant, measurements, {"k_hyd_ch": default_k})

print("channels returned:", sorted(predicted))

observed = measurements.data["Q_gas"].to_numpy()
simulated = np.asarray(predicted["Q_gas"], dtype=float)
bias = simulated.mean() / observed.mean() - 1.0
print(f"\nmeasured  mean Q_gas: {observed.mean():8.1f} m3/d")
print(f"simulated mean Q_gas: {simulated.mean():8.1f} m3/d   ({bias:+.1%})")

## Display Measurement and Simulation Values

In [ ]:
channels = ["Q_gas", "Q_ch4", "P_el", "pH", "VFA"]
units = {"Q_gas": "m3/d", "Q_ch4": "m3/d", "P_el": "kW", "pH": "-", "VFA": "g/L"}

fig, axes = plt.subplots(3, 2, figsize=(11, 7), sharex=True)
for ax, channel in zip(axes.ravel(), channels):
    measured = measurements.data[channel].to_numpy()
    modelled = np.asarray(predicted[channel], dtype=float)
    ax.plot(measurements.data.index, measured, label="measured", alpha=0.7)
    ax.plot(measurements.data.index, modelled, label="simulated")
    ax.set_title(f"{channel} [{units[channel]}]", fontsize=10)
    ax.tick_params(axis="x", rotation=30)

axes.ravel()[-1].axis("off")
axes[0, 0].legend(loc="lower right", fontsize=8)
fig.suptitle("Every measured channel, model against measurement")
fig.tight_layout()

Three of the five channels carry the same error of roughly four percent -
`Q_gas`, `Q_ch4` and `P_el`. That is no coincidence: methane is a share of the
gas, and the CHP turns that gas into electricity, so one modelling error
propagates through all three. They are three views of one problem, not three
problems.

`pH`, on the other hand, deviates by **0.03 %**. Calibrating against pH would
therefore tell you almost nothing here: the parameter that is wrong barely moves
that channel. Picking the channel is as much a decision as picking the
parameter.